# 10. Feature selection (race and sex)

Same procedure as `04_feature_selection.ipynb` applied to the preliminary race and sex models of
`09_race_sex_preliminary.ipynb` (same target-specific exclusions): variables are ranked by M1/M3/M4 and the models are
retrained with the top N = 10, 20, …, 90 variables.

The feature lists of the race and sex models in Figure 5 (`inputs/top_40_features_m3_race.xlsx` and
`inputs/top_40_features_m3_sex.xlsx`, sheets `a_XGB`, `b_XGB`, `c_XGB`) are the M3 top 40 of each ballot
(`6_1_top_features_long.xlsx`, method M3, top_n 40) with the same manual adjustments as the party identity model:
`zodiac` was removed, and `relig` was removed where it overlapped with `reltrad` (race model);
removed variables were replaced by the next-ranked variables.

**Input**: `data/5_cleaned_data_for_{race,sex}_analysis.pkl`,
`outputs/5_1_preliminary_race_sex_noatt/5_1_Preliminary_NoAtt_{race_binary,sex}_analysis_all_importance_weighted.parquet`

**Output**: `outputs/6_1_feature_selection_race_sex_noatt/` (`6_1_top_features_long.{xlsx,csv}` and `6_1_FS_*` results)

## Excluded variables

In [ ]:
import pandas as pd
import numpy as np
import shap
import pickle
import os
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb

# ============================================================================
# Target-specific excluded variables (same principle as the party identity models)
#   race model: only direct/indirect measures of race are excluded (sex variables stay as features)
#   sex  model: only direct/indirect measures of sex are excluded (race variables stay as features)
# ============================================================================
RACE_POTENTIAL_TARGETS = [
    'race', 'race_binary', 'racecen1', 'racecen2', 'racecen3', 'hispanic', 'hisporig', 'hhrace',
    'racecen1_0022', 'racecen2_0022', 'racecen3_0022', 'hispanic_0022', 'racerank1', 'racerank2', 'racerank3', 'raceacs1',
    'raceacs2', 'raceacs3', 'raceacs4', 'raceacs5', 'raceacs6', 'raceacs7', 'raceacs8', 'raceacs9',
    'raceacs10', 'raceacs11', 'raceacs12', 'raceacs13', 'raceacs14', 'raceacs15', 'raceacs16', 'raceacs14_2122',
    'ethnic', 'eth1', 'ethworld1', 'ethworld2', 'ethworld3', 'ethworld4', 'ethworld5', 'ethworld6',
    'ethworld7', 'ethworld8', 'ethworld9', 'ethregion1', 'ethregion2', 'ethregion3', 'ethregion4', 'ethregion5',
    'ethregion6', 'ethregion7', 'ethregion8', 'ethregion9', 'ethregion10', 'ethregion11', 'ethregion12', 'ethregion13',
    'ethregion14', 'ethregion15', 'ethregion16', 'ethregion17', 'ethregion18', 'ethregion19', 'ethregion20', 'ethregion21',
    'ethregion22', 'ethregion23', 'ethregion24', 'ethregion25', 'ethregion26', 'ethregion27', 'ethregion28', 'ethregion29',
    'ethregion30', 'ethregion31', 'ethregion32', 'ethregion33', 'ethregion34', 'ethregion35', 'ethregion36', 'ethregion37',
    'ethregion38', 'ethregion39', 'ethregion40', 'ethregion41', 'ethregion42', 'ethregion43', 'ethregion44', 'ethregion45',
    'ethregion46', 'ethregion47', 'ethregion48', 'ethregion49', 'ethregion50', 'ethregion51', 'ethregion52', 'ethregion53',
    'ethregion54', 'ethregion55', 'ethregion56', 'ethregion57', 'ethregion58', 'ethregion59', 'ethregion60', 'ethregion61',
    'ethregion62', 'ethregion63', 'ethregion64', 'ethregion65', 'ethregion66', 'ethregion67', 'ethregion68', 'ethregion69',
    'ethregion70', 'ethregion71', 'ethregion72', 'ethregion73', 'ethregion74', 'ethregion75', 'ethregion76', 'ethregion77',
    'ethregion78', 'ethregion79', 'ethregion80', 'ethregion81', 'ethregion82', 'ethregion83', 'ethregion84', 'ethregion85',
    'ethregion86', 'ethregion87', 'ethregion88', 'ethregion89', 'ethregion90', 'ethregion91', 'ethregion92', 'ethregion93',
    'ethregion94', 'ethregion95', 'racesee', 'racedbtf', 'racethwh', 'racethbl', 'racethhi', 'racethas',
    'racethna', 'racethot', 'ethimp', 'ethnum', 'chldfam_wht', 'chldfam_blk', 'chldfam_asn', 'chldfam_hisp',
    'chldfam_oth', 'closeblk', 'marblk', 'liveblks',
]
SEX_POTENTIAL_TARGETS = [
    'sex', 'sexbirth1', 'sexnow1', 'sexnow2', 'gender1', 'rplace', 'rplace_7522', 'femsee1',
    'femself1', 'mascsee1', 'mascself1', 'rheight',
]
NOT_NEEDED = [
    'hlthstrt', 'huadd', 'huaddwhy', 'dwellpre', 'kidsinhh', 'respond', 'incuspop', 'neisafe',
    'rlooks', 'rgroomed', 'rweight', 'rhlthend', 'huclean', 'coop', 'comprend', 'consent',
    'adminconsent', 'whoelse1', 'whoelse2', 'whoelse3', 'whoelse4', 'whoelse5', 'whoelse6', 'lngthinv',
    'intage', 'intage_0418', 'intethn', 'intsex', 'intyrs', 'intyrs_0418', 'inthisp', 'intrace1',
    'intrace2', 'intrace3', 'ratetone', 'mode', 'modesequence', 'feeused', 'feelevel', 'totalincentive',
    'spaneng', 'spanint', 'spanself', 'formwt', 'oversamp', 'wtss', 'wtssall', 'wtssnr',
    'wtsps', 'wtssps', 'wtssnrps', 'ballotformwt', 'ballotformwtnr', 'wtsps_as', 'wtssnrps_as', 'id',
    'hefinfo', 'hefinfo1', 'respnum', 'respnumh', 'year', 'dateintv', 'ballot', 'version',
    'form', 'issp', 'phone', 'sample', 'sampcode', 'phase', 'subsamprate', 'batch',
    'kish', 'vpsu', 'vstrat', 'fileversion', 'final_weight', 'intid', 'genevig1', 'genevig2',
    'mostimp1', 'mostimp2', 'mostimp3', 'mostimp4', 'prob1', 'prob2', 'prob3', 'prob4',
    'extra1', 'extra2', 'srcbelt', 'size', 'babies_7222', 'earnrs_7222', 'hompop_7222', 'preteen_7222',
    'reg16_7222', 'region_7222', 'sibs_7222', 'teens_7222', 'saqsex', 'saqissp', 'easyget', 'scimode',
    'sciintro', 'rundrstd', 'robject', 'intrwght', 'saqgene', 'coop2', 'hhtype',
]

ATTITUDE_RACE = [
    'racdif1', 'racdif2', 'racdif3', 'racdif4', 'racdif5', 'racdif1_c', 'racdif2_c', 'racdif3_c',
    'racdif4_c', 'racdif5_c', 'racdif1x', 'racdif2x', 'racdif3x', 'racdif4x', 'racdif5x', 'natrace',
    'natrace_c', 'natracey', 'natracez', 'discaff', 'discaff_c', 'discaffy', 'affrmact', 'affrmact2',
    'helpblk', 'wrkwayup', 'busing', 'feelblks', 'feelwhts', 'closewht', 'racopen', 'racopen_c',
    'racmar', 'racdin', 'racpush', 'racseg', 'racfew', 'rachaf', 'racmost', 'raclive',
    'racpres', 'racschol', 'racwork', 'racchurh', 'racinteg', 'racchng', 'dsndntpay', 'workblks',
    'workwhts', 'intlblks', 'intlwhts', 'wlthblks', 'wlthwhts', 'fareblks', 'farewhts', 'rachome',
    'marwht', 'livewhts', 'blkwhite', 'blkjobs', 'blkhouse', 'blksimp', 'whtsdiff',
]
ATTITUDE_SEX = [
    'fepol', 'fepol_c', 'fepoly', 'fework', 'fechld', 'fechld2', 'fepresch', 'fepresch2',
    'fefam', 'fehome', 'fehelp', 'fehire', 'fejobaff', 'meovrwrk', 'discaffw', 'discaffw_c',
    'discaffm', 'homosex', 'homosex1', 'marhomo', 'premarsx', 'teensex', 'xmarsex', 'xmarsex1',
    'pillok', 'sexeduc', 'feovrwrk', 'hubbywrk', 'hubbywk', 'hubbywk1', 'twoincs', 'twoincs1',
    'gendereq', 'fepres',
]

def expand_to_ballot_model(ballot_keyed, models=('XGBoost',)):
    """{ballot: {year: df}} -> {(ballot, model): {year: df}} (same key structure as the party identity notebooks).
    The same year_dict is shared by reference; the model functions work on .copy(), so this is safe."""
    expanded = {}
    for ballot, year_dict in ballot_keyed.items():
        for model in models:
            expanded[(ballot, model)] = year_dict
    return expanded

## Variable rankings (M1, M3, M4)

In [ ]:
def get_top_features_method1(all_importances_df, top_n_list):
    """
    Method 1: top-N variable list per (ballot, model)
    - The same (ballot, model) uses the same variable list in all years.
    """
    print("--- Method 1: top-N variable lists per (ballot, model) ---")

    # Mean importance over all years and runs
    avg_imp = all_importances_df.groupby(['ballot', 'model', 'feature'])['importance'].mean().reset_index()

    top_features_all_n = {}
    for n in top_n_list:
        # Top N per group
        top_n_features = avg_imp.sort_values('importance', ascending=False).groupby(['ballot', 'model']).head(n)

        feature_dict = {}
        # Dictionary keyed by (ballot, model)
        for (ballot, model), group in top_n_features.groupby(['ballot', 'model']):
            feature_dict[(ballot, model)] = group['feature'].tolist()

        top_features_all_n[n] = feature_dict
        print(f"Top {n}: variable lists for {len(feature_dict)} (ballot, model) groups.")

    return top_features_all_n

def get_top_features_method3(all_importances_df, top_n_list, top_n_threshold=20):
    """
    Method 3: returns { N: { (ballot, model): [feature_list] } }
    - Criterion: raw count of (run, year, model) tasks in which the variable is in the top 20
    """
    print(f"--- Method 3: top-N variable lists per (ballot, model) by raw top-{top_n_threshold} frequency ---")

    importance_df_cleaned = all_importances_df.dropna(subset=['importance'])
    required_cols = ['run', 'ballot', 'year', 'model', 'feature', 'importance']
    if not all(col in importance_df_cleaned.columns for col in required_cols):
        print(f"!! Error: input DataFrame lacks required columns ({required_cols}).")
        return {}
    if importance_df_cleaned.empty:
        print("!! Warning: no valid importance data.")
        return {}

    top_n_group_keys = ['run', 'ballot', 'year', 'model']
    top_features_per_instance = importance_df_cleaned.sort_values('importance', ascending=False)\
                                                     .groupby(top_n_group_keys)\
                                                     .head(top_n_threshold)

    frequency_counts = top_features_per_instance.groupby(['ballot', 'model', 'feature'])\
                                                .size()\
                                                .reset_index(name='frequency_top_n')


    top_features_all_n = {} # { N: { (ballot, model): [...] } }
    for n in top_n_list:
        feature_dict_for_n = {} # { (ballot, model): [...] } for this N
        # Ties at the top-N cutoff are broken by pandas' default (unstable) sort, so the order of
        # tied variables depends on the full input table. The published lists were computed from an
        # importance table that also held LightGBM rows. With XGBoost-only input some lists differ
        # from the published ones: most only in the order of variables, the rest by swapping
        # variables tied at the cutoff.
        top_n_final = frequency_counts.sort_values('frequency_top_n', ascending=False)\
                                      .groupby(['ballot', 'model'])\
                                      .head(n)

        for (ballot, model), group in top_n_final.groupby(['ballot', 'model']):
            # Ballot as string for stable keys
            feature_dict_for_n[(str(ballot), model)] = group['feature'].tolist()

        top_features_all_n[n] = feature_dict_for_n
        print(f"Method 3 - Top {n}: variable lists for {len(feature_dict_for_n)} (ballot, model) groups.")


    return top_features_all_n


def get_top_features_method4(all_importances_df, top_n_list, top_n_threshold=20):
    """
    Method 4: returns { N: { (ballot, model): [feature_list] } }
    - Criterion: top-20 frequency of (run, year, model) tasks normalized by the number of tasks using the variable
    """
    print(f"--- Method 4: top-N variable lists per (ballot, model) by normalized top-{top_n_threshold} frequency ---")

    importance_df_cleaned = all_importances_df.dropna(subset=['importance'])
    required_cols = ['run', 'ballot', 'year', 'model', 'feature', 'importance']
    if not all(col in importance_df_cleaned.columns for col in required_cols):
        print(f"!! Error: input DataFrame lacks required columns ({required_cols}).")
        return {}
    if importance_df_cleaned.empty:
        print("!! Warning: no valid importance data.")
        return {}

    top_n_group_keys = ['run', 'ballot', 'year', 'model']
    top_features_per_instance = importance_df_cleaned.sort_values('importance', ascending=False)\
                                                     .groupby(top_n_group_keys)\
                                                     .head(top_n_threshold)

    frequency_numerator = top_features_per_instance.groupby(['ballot', 'model', 'feature'])\
                                                   .size()\
                                                   .reset_index(name='frequency_top_n')

    frequency_denominator = importance_df_cleaned.groupby(['ballot', 'model', 'feature'])\
                                                 .size()\
                                                 .reset_index(name='total_occurrences')

    frequency_normalized_df = pd.merge(
        frequency_numerator,
        frequency_denominator,
        on=['ballot', 'model', 'feature'],
        how='left'
    )
    frequency_normalized_df['total_occurrences'] = frequency_normalized_df['total_occurrences'].replace(0, 1)
    frequency_normalized_df['frequency_normalized'] = frequency_normalized_df['frequency_top_n'].astype(float) / frequency_normalized_df['total_occurrences']
    frequency_normalized_df.fillna({'frequency_normalized': 0}, inplace=True)


    top_features_all_n = {} # { N: { (ballot, model): [...] } }
    for n in top_n_list:
        feature_dict_for_n = {} # { (ballot, model): [...] } for this N
        # Ties at the top-N cutoff are broken by pandas' default (unstable) sort, so the order of
        # tied variables depends on the full input table. The published lists were computed from an
        # importance table that also held LightGBM rows. With XGBoost-only input some lists differ
        # from the published ones: most only in the order of variables, the rest by swapping
        # variables tied at the cutoff.
        top_n_final = frequency_normalized_df.sort_values('frequency_normalized', ascending=False)\
                                             .groupby(['ballot', 'model'])\
                                             .head(n)

        for (ballot, model), group in top_n_final.groupby(['ballot', 'model']):
            # Ballot as string for stable keys
            feature_dict_for_n[(str(ballot), model)] = group['feature'].tolist()

        top_features_all_n[n] = feature_dict_for_n
        print(f"Method 4 - Top {n}: variable lists for {len(feature_dict_for_n)} (ballot, model) groups.")


    return top_features_all_n

## Model training with selected features (one run)

In [ ]:
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def run_single_ml_iteration_weighted_selective(
    data_for_analysis_dict,  # {(ballot, model): {year: df}} returned by build_selective_data_dict
    target_col='partyid_1',
    random_seed=42,
    run_number=1,
    use_weights=True
):
    """
    Run 5-fold CV on the data already restricted to the selected features
    by build_selective_data_dict and return the SHAP importance.
    """
    scores_list = []
    importances_list = []
    dimensions_list = []
    feature_usage_list = []

    for (ballot, model_name), year_dict in data_for_analysis_dict.items():

        # An unknown model_name means the upstream data structure is broken,
        # so stop immediately.
        if model_name not in ['XGBoost']:
            raise ValueError(f"Unknown model_name: '{model_name}' (ballot={ballot})")

        for year, df_original in year_dict.items():
            dataset_identifier = f"({ballot}, {model_name}) - {year}"

            df = df_original.copy()
            if not df.index.is_unique:
                df.reset_index(drop=True, inplace=True)

            # Upstream steps should only create non-empty data that contain target_col.
            # A violation indicates an upstream bug, so stop.
            if df.empty:
                raise ValueError(f"{dataset_identifier} - empty data frame.")
            if target_col not in df.columns:
                raise ValueError(f"{dataset_identifier} - target column '{target_col}' not found.")

            target_series = df[target_col]
            if isinstance(target_series, pd.DataFrame):
                target_series = target_series.iloc[:, 0]
            df_filtered = df[target_series.notna() & (target_series != 0)].copy()

            # No cases after filtering means the year/ballot data must be checked.
            if df_filtered.empty:
                raise ValueError(f"{dataset_identifier} - no cases left after filtering '{target_col}'.")
            df_filtered.reset_index(drop=True, inplace=True)

            y_original = df_filtered[target_col]
            if isinstance(y_original, pd.DataFrame):
                y_original = y_original.iloc[:, 0]

            # GSS weights: wtssnrps from 2004 on, wtssps before; equal weights (1) if neither exists
            weights_original = pd.Series(np.ones(len(df_filtered)), index=df_filtered.index)
            if use_weights:
                if 'wtssnrps' in df_filtered.columns and pd.to_numeric(year, errors='coerce') >= 2004:
                    weights_original = df_filtered['wtssnrps']
                elif 'wtssps' in df_filtered.columns:
                    weights_original = df_filtered['wtssps']
                weights_original = weights_original.fillna(1)

            # 5-fold CV needs at least 20 cases and 2 classes.
            if len(df_filtered) < 20 or y_original.nunique() < 2:
                raise ValueError(
                    f"{dataset_identifier} - too few cases ({len(df_filtered)}) or "
                    f"too few classes of '{target_col}' ({y_original.nunique()})."
                )

            # potential_targets/not_needed are already handled in build_selective_data_dict;
            # only the target and weight columns are removed here.
            meta_cols = [target_col, 'wtssnrps', 'wtssps']
            features_to_drop = [col for col in meta_cols if col in df_filtered.columns]
            X = df_filtered.drop(columns=features_to_drop, errors='ignore')

            # List-wise deletion: drop cases with a missing value in any
            # of the variables used in the model.
            X = X.dropna()
            y_original = y_original.loc[X.index]
            weights_original = weights_original.loc[X.index]

            # Re-check the number of cases/classes after list-wise deletion.
            # Skip with a warning when data are too thin (e.g. ballot rotation).
            if len(y_original) < 20 or y_original.nunique() < 2:
                print(
                    f"[SKIP] {dataset_identifier} - too few cases ({len(y_original)}) or "
                    f"classes ({y_original.nunique()}) after list-wise deletion."
                )
                continue

            feature_names = X.columns.tolist()

            # No features left means a problem in build_selective_data_dict or feature_dict.
            if not feature_names:
                raise ValueError(f"{dataset_identifier} - no features (all columns excluded).")

            feature_usage_list.append({
                'run': run_number, 'ballot': ballot, 'year': year,
                'model': model_name, 'features_used': ','.join(feature_names)
            })

            y_encoded = LabelEncoder().fit_transform(y_original)
            n_classes = len(np.unique(y_encoded))

            dimensions_list.append({
                'run': run_number, 'ballot': ballot, 'year': year,
                'model': model_name, 'n_rows': len(X), 'n_cols': len(feature_names)
            })

            model = xgb.XGBClassifier(
                eval_metric='mlogloss', random_state=random_seed, enable_categorical=True
            )

            kfold = KFold(n_splits=5, shuffle=True, random_state=random_seed)
            fold_scores = {'accuracy': [], 'f1_macro': [], 'roc_auc_ovr': []}
            fold_importances = []

            for fold_i, (train_idx_rel, test_idx_rel) in enumerate(kfold.split(X, y_encoded)):
                train_abs = X.index[train_idx_rel]
                test_abs  = X.index[test_idx_rel]
                X_train, X_test = X.loc[train_abs], X.loc[test_abs]
                y_train, y_test = y_encoded[train_idx_rel], y_encoded[test_idx_rel]
                w_train, w_test = weights_original.loc[train_abs], weights_original.loc[test_abs]

                fit_params = {'sample_weight': w_train} if use_weights else {}
                model.fit(X_train, y_train, **fit_params)

                y_pred  = model.predict(X_test)
                y_proba = model.predict_proba(X_test)
                fold_scores['accuracy'].append(accuracy_score(y_test, y_pred, sample_weight=w_test))
                fold_scores['f1_macro'].append(f1_score(y_test, y_pred, average='macro', sample_weight=w_test))
                if n_classes > 2:
                    fold_scores['roc_auc_ovr'].append(
                        roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro', sample_weight=w_test))
                else:
                    fold_scores['roc_auc_ovr'].append(
                        roc_auc_score(y_test, y_proba[:, 1], average='macro', sample_weight=w_test))

                if X_test.empty:
                    fold_importances.append(np.array([np.nan] * len(feature_names)))
                    continue

                explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
                shap_out = explainer.shap_values(X_test, check_additivity=False)

                if isinstance(shap_out, list):
                    # A list of SHAP arrays must have one element per class.
                    if len(shap_out) != n_classes:
                        raise ValueError(
                            f"{dataset_identifier} fold {fold_i} - length of SHAP list ({len(shap_out)}) "
                            f"does not match the number of classes ({n_classes})."
                        )
                    shap_arr = shap_out[1]
                elif isinstance(shap_out, np.ndarray):
                    shap_arr = shap_out
                else:
                    raise ValueError(
                        f"{dataset_identifier} fold {fold_i} - unexpected SHAP output type: {type(shap_out)}"
                    )

                mean_abs = np.abs(shap_arr).mean(axis=0)
                # SHAP dimensions must match the number of features.
                if len(mean_abs) != len(feature_names):
                    raise ValueError(
                        f"{dataset_identifier} fold {fold_i} - SHAP dimension ({len(mean_abs)}) "
                        f"does not match the number of features ({len(feature_names)})."
                    )
                fold_importances.append(mean_abs)

            # Collect fold results
            for metric, values in fold_scores.items():
                if values:
                    scores_list.append({
                        'ballot': ballot, 'model': model_name, 'year': year, 'run': run_number,
                        'metric': metric, 'avg_score': np.mean(values), 'std_score': np.std(values)
                    })

            if fold_importances:
                avg_imp = np.nanmean(np.array(fold_importances), axis=0)
                for feat, imp in zip(feature_names, avg_imp):
                    if not np.isnan(imp):
                        importances_list.append({
                            'ballot': ballot, 'model': model_name, 'year': year, 'run': run_number,
                            'feature': feat, 'importance': float(imp)
                        })

    return (
        pd.DataFrame(scores_list), pd.DataFrame(importances_list),
        pd.DataFrame(dimensions_list), pd.DataFrame(feature_usage_list)
    )

## Data subsetting, repeated runs, summaries and saving

In [ ]:
import os

# Output folder of this notebook
OUT_DIR = "outputs/6_1_feature_selection_race_sex_noatt"

def build_selective_data_dict(df_dict, feature_dict, target_col):
    """
    Read the selected features of each (ballot, model) from feature_dict and
    restrict every yearly data frame in df_dict to those features.

    potential_targets/not_needed should already be excluded when feature_dict is built,
    so the exclusion here is a second safety net
    against leakage variables included in feature_dict by mistake.
    """
    # Excluded variables by target (direct/indirect measures of race/sex; safety net against leakage)
    if target_col == 'race_binary':
        potential_targets = RACE_POTENTIAL_TARGETS + ATTITUDE_RACE
    elif target_col == 'sex':
        potential_targets = SEX_POTENTIAL_TARGETS + ATTITUDE_SEX
    else:
        raise ValueError(f"Unsupported target_col: '{target_col}' (race_binary or sex)")
    exclusion_base = set(potential_targets + NOT_NEEDED)

    result = {}
    for (ballot, model_name), year_dict in df_dict.items():
        # M1 uses (ballot, model) keys; M3/M4 may use (str(ballot), model) keys.
        key = (ballot, model_name)
        if key not in feature_dict:
            key = (str(ballot), model_name)
        # A missing key means feature_dict is malformed, so stop.
        if key not in feature_dict:
            raise KeyError(
                f"Key not found in feature_dict: ({ballot}, {model_name}) "
                f"or ({str(ballot)}, {model_name})"
            )
        selective_features = feature_dict[key]

        year_result = {}
        for year, df in year_dict.items():
            existing = [f for f in selective_features if f in df.columns and f not in exclusion_base]

            # Skip years that contain none of the selected features
            # (expected, because questionnaires differ across years).
            if not existing:
                continue

            # Keep target_col and weight columns for the model functions
            weight_cols = [w for w in ['wtssnrps', 'wtssps'] if w in df.columns]
            cols_to_keep = existing + ([target_col] if target_col in df.columns else []) + weight_cols

            year_result[year] = df[cols_to_keep].copy()

        result[(ballot, model_name)] = year_result
    return result


def run_ml_multiple_times_weighted_selective(
    data_for_analysis_dict,  # {(ballot, model): {year: df}} returned by build_selective_data_dict
    target_col='partyid_1',
    n_runs=3,
    use_weights=True
):
    """
    Run run_single_ml_iteration_weighted_selective n_runs times and combine the results.
    """
    all_scores_list = []
    all_importances_list = []
    all_dimensions_list = []
    all_feature_usage_list = []

    for i in range(n_runs):
        print(f"\n{'='*30}\n         RUN {i+1} / {n_runs}         \n{'='*30}")

        run_scores_df, run_importances_df, run_dimensions_df, run_feature_usage_df = \
            run_single_ml_iteration_weighted_selective(
                data_for_analysis_dict=data_for_analysis_dict,
                target_col=target_col,
                random_seed=i,
                run_number=i + 1,
                use_weights=use_weights
            )

        all_scores_list.append(run_scores_df)
        all_importances_list.append(run_importances_df)
        all_dimensions_list.append(run_dimensions_df)
        all_feature_usage_list.append(run_feature_usage_df)
        print(f"--- Run {i+1} completed. ---")

    scores_to_concat      = [df for df in all_scores_list       if isinstance(df, pd.DataFrame) and not df.empty]
    importances_to_concat = [df for df in all_importances_list  if isinstance(df, pd.DataFrame) and not df.empty]
    dims_to_concat        = [df for df in all_dimensions_list   if isinstance(df, pd.DataFrame) and not df.empty]
    usage_to_concat       = [df for df in all_feature_usage_list if isinstance(df, pd.DataFrame) and not df.empty]

    # No scores from any run means a problem in the input data or the functions above.
    if not scores_to_concat:
        raise RuntimeError("No scores were produced in any run. Check the input data and function definitions.")

    all_scores_df        = pd.concat(scores_to_concat,      ignore_index=True)
    all_importances_df   = pd.concat(importances_to_concat, ignore_index=True) if importances_to_concat else pd.DataFrame()
    all_dimensions_df    = pd.concat(dims_to_concat,        ignore_index=True) if dims_to_concat        else pd.DataFrame()
    all_feature_usage_df = pd.concat(usage_to_concat,       ignore_index=True) if usage_to_concat       else pd.DataFrame()

    # Fails loudly if all_scores_df does not have the expected columns.
    final_summary = all_scores_df.groupby(['ballot', 'year', 'model', 'metric'])['avg_score'].agg(['mean', 'std']).reset_index()
    final_summary = final_summary.rename(columns={'mean': 'final_avg_score', 'std': 'final_std_score'})
    print(f"\n\n{'='*40}\n     FINAL AVERAGED RESULTS ({n_runs} RUNS)     \n{'='*40}")
    print(final_summary[final_summary['metric'] == 'roc_auc_ovr'].round(4))

    return all_scores_df, all_importances_df, final_summary, all_dimensions_df, all_feature_usage_df


def compare_model_performance(scores_df, metric='roc_auc_ovr'):
    """Mean/SD of `metric` by (ballot, year, model), sorted."""
    print(f"\n{'='*30}\n   Model performance ({metric})   \n{'='*30}")

    if scores_df.empty or metric not in scores_df['metric'].values:
        print("!! Warning: Scores DataFrame empty/metric not found.")
        return pd.DataFrame(columns=['ballot', 'year', 'model', 'avg_score_mean', 'avg_score_std'])

    metric_df = scores_df[scores_df['metric'] == metric].copy()
    performance_summary = metric_df.groupby(['ballot', 'year', 'model'])['avg_score'].agg(['mean', 'std']).reset_index()
    performance_summary = performance_summary.rename(columns={'mean': 'avg_score_mean', 'std': 'avg_score_std'})
    performance_summary = performance_summary.sort_values(by=['ballot', 'year', 'avg_score_mean'], ascending=[True, True, False])
    print(performance_summary.round(4))
    return performance_summary





def analyze_feature_frequency(importance_df, top_n=5):
    """
    For each (ballot, model, run, year), take the top_n features by importance and count
    how often each feature is in the top N (frequency_top_n) and how often it is used at all (total_occurrences);
    frequency_normalized = frequency_top_n / total_occurrences.
    A high ratio means that whenever the feature is used in that (ballot, model) group,
    it almost always ranks near the top.
    """

    # --- 0. Denominator base ---
    # With the {(ballot, model): {year: df}} structure, 'ballot' and 'year' columns always exist.
    denominator_base_df = importance_df.copy()
    top_n_group_keys = ['run', 'ballot', 'year', 'model']

    # --- 1. Top-N features (numerator base) ---
    # (rows used for 'frequency_top_n')
    top_features = importance_df.sort_values('importance', ascending=False).groupby(top_n_group_keys).head(top_n)

    numerator_base_df = top_features

    # --- 3. Overall frequency (not split by ballot/model) ---
    # Counts only; no normalization because groups are not split.
    overall_analysis = top_features.groupby('feature')['importance'].agg(
        frequency=('count'),
        importance_mean=('mean')
    ).reset_index().sort_values(by='frequency', ascending=False)

    # --- 4. By group: numerator ---
    grouped_analysis_numerator = numerator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        frequency_top_n=('count'), # (A) times in the top N
        importance_mean=('mean')
    ).reset_index()

    # --- 5. By group: denominator ---
    grouped_analysis_denominator = denominator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        total_occurrences=('count') # (B) times the variable was used
    ).reset_index()

    # --- 6. Merge (A) and (B), normalized frequency ---
    grouped_analysis = pd.merge(
        grouped_analysis_numerator,
        grouped_analysis_denominator,
        on=['ballot', 'model', 'feature'],
        how='left' # keyed on the variables in the top N
    )

    # (A) / (B)
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_top_n'].astype(float) / grouped_analysis['total_occurrences']
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_normalized'].fillna(0) # guard against NaN

    # --- 7. Sort by normalized frequency ---
    grouped_analysis = grouped_analysis.sort_values(
        by=['ballot', 'model', 'frequency_normalized'],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    # --- 8. Print results by (ballot, model) ---
    print("\n--- Grouped Frequency Analysis (Normalized) ---")
    for name, group in grouped_analysis.groupby(['ballot', 'model']):
        ballot_name, model_name = name
        print(f"\n[Group: {ballot_name} - {model_name}]")
        # Columns to show
        print(group[[
            'feature',
            'frequency_normalized',
            'frequency_top_n',
            'total_occurrences',
            'importance_mean'
        ]].head(5).to_markdown(index=False, floatfmt=".4f"))

    return overall_analysis, grouped_analysis





def analyze_temporal_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by year (score + rank)   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping temporal analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for temporal analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_model = {}
    for model_name in avg_importance_df['model'].unique():
        model_df = avg_importance_df[avg_importance_df['model'] == model_name]
        for ballot_type in sorted(model_df['ballot'].unique()):
            ballot_df = model_df[model_df['ballot'] == ballot_type]
            # A failing pivot (e.g. duplicated (feature, year) pairs) means importance_df
            # is malformed, so the exception is allowed to stop the analysis.
            pivot_table = ballot_df.pivot_table(index='feature', columns='year', values=['importance', 'rank'])
            pivot_table = pivot_table.swaplevel(0, 1, axis=1).sort_index(axis=1)
            results_by_model[f"Temporal_{model_name}_{ballot_type}"] = pivot_table
    return results_by_model





def analyze_cross_model_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by model and year   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping cross-model analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for cross-model analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_ballot = {}
    for ballot_type in sorted(avg_importance_df['ballot'].unique()):
        ballot_df = avg_importance_df[avg_importance_df['ballot'] == ballot_type]
        # A failing pivot (e.g. duplicated (feature, year, model) triples) means importance_df
        # is malformed, so the exception is allowed to stop the analysis.
        pivot_table = ballot_df.pivot_table(index='feature', columns=['year', 'model'], values=['importance', 'rank'])
        pivot_table.columns = [f"{year}_{model}_{val}" for val, year, model in pivot_table.columns]
        pivot_table = pivot_table.sort_index(axis=1)
        results_by_ballot[f"CrossModel_{ballot_type}"] = pivot_table
    return results_by_ballot





def save_results_to_excel(results_dict, filename="ml_analysis_summary.xlsx"):
    """Save each item of results_dict as one sheet of an Excel file."""
    with pd.ExcelWriter(filename) as writer:
        for sheet_name, df in results_dict.items():
             if isinstance(df, pd.DataFrame):
                # Ensure sheet names are valid
                safe_sheet_name = str(sheet_name).replace('[','').replace(']','').replace('*','').replace(':','').replace('?','').replace('/','\\')
                safe_sheet_name = safe_sheet_name[:31] # Max length limit

                if isinstance(df.index, (pd.MultiIndex, pd.Index)) and df.index.name is not None:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=True)
                else:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
    print(f"\n  All results saved to '{filename}'.")


# --- Full workflow: training -> performance/importance summaries -> Excel/Parquet ---




def run_analysis_for_target_weighted_selective(
    df_dict,    # {(ballot, model): {year: df}} returned by build_selective_data_dict
    df_name,    # name used in the output file names (e.g. "2_Feature Selection_M3_Top30")
    target_col,
    n_runs
):
    """
    Run the full retraining workflow for one target variable and save the results:
    run_ml_multiple_times_weighted_selective -> summary functions -> Excel/Parquet.
    """
    print(f"\n{'='*60}")
    print(f"  Starting Selective Workflow for Target: '{target_col}' ({n_runs} runs)")
    print(f"  Data Source: {df_name}")
    print(f"{'='*60}")

    all_scores, all_importances, final_summary, all_dimension, all_feature_usage = \
        run_ml_multiple_times_weighted_selective(
            data_for_analysis_dict=df_dict,
            target_col=target_col,
            n_runs=n_runs,
            use_weights=True
        )

    performance_comparison_df = compare_model_performance(all_scores, metric='roc_auc_ovr')
    overall_freq_df, grouped_freq_df = analyze_feature_frequency(all_importances, top_n=20)
    temporal_results    = analyze_temporal_change(all_importances)
    cross_model_results = analyze_cross_model_change(all_importances)

    print(f"\n--- [{target_col} - {df_name}] Collecting results... ---")

    # Fails loudly if all_dimension does not have the expected columns.
    avg_dims = all_dimension.groupby(['ballot', 'year', 'model'])[['n_rows', 'n_cols']].mean().reset_index()

    unique_feature_usage_df = (
        all_feature_usage[all_feature_usage['run'] == 1]
        .drop(columns=['run'])
        .reset_index(drop=True)
    )

    excel_sheets = {
        "0_Final_Summary":       final_summary,
        "1_Dataset_Dimensions":  avg_dims,
        "2_Performance_Compare": performance_comparison_df,
        "3_Feature_Freq_Overall": overall_freq_df,
        "4_Feature_Freq_Grouped": grouped_freq_df,
        "5_Feature_Usage_Log":   unique_feature_usage_df,
        **temporal_results,
        **cross_model_results
    }

    os.makedirs(OUT_DIR, exist_ok=True)
    excel_filename  = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_results_weighted.xlsx")
    import_filename = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_all_importance_weighted.parquet")
    scores_filename = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_all_scores_weighted.parquet")
    usage_filename  = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_feature_usage_log.parquet")

    save_results_to_excel(excel_sheets, filename=excel_filename)

    print(f"\n--- [{target_col} - {df_name}] Saving parquet files... ---")
    if not all_importances.empty:   all_importances.to_parquet(import_filename)
    if not all_scores.empty:        all_scores.to_parquet(scores_filename)
    if not all_feature_usage.empty: all_feature_usage.to_parquet(usage_filename)

    print(f"  [{target_col} - {df_name}] All files saved.")
    print(f"{'-'*60}\n")

In [ ]:
def load_pickle_data(filename):
    """Load a pickle file."""
    print(f"--- Loading cleaned data from '{filename}'... ---")
    try:
        with open(filename, 'rb') as f:
            data = pickle.load(f)
        print("Loaded.")
        return data
    except FileNotFoundError:
        print(f"!! Error: {filename} not found. Run the previous notebooks first.")
        return None

def load_parquet_data(filename):
    """Load a parquet file."""
    print(f"--- Loading importance data from '{filename}'... ---")
    try:
        data = pd.read_parquet(filename)
        print("Loaded.")
        return data
    except Exception as e:
        print(f"!! Error: failed to load {filename}: {e}")
        print("Check that the parquet file of the preliminary model exists.")
        return pd.DataFrame()

## Load the preliminary results and build the rankings

In [ ]:
TOP_N_LIST = [10, 20, 30, 40, 50, 60, 70, 80, 90]

# Convert to the {(ballot, model): {year: df}} key structure
data_race = expand_to_ballot_model(load_pickle_data('data/5_cleaned_data_for_race_analysis.pkl'))
data_sex  = expand_to_ballot_model(load_pickle_data('data/5_cleaned_data_for_sex_analysis.pkl'))

imp_race = load_parquet_data('outputs/5_1_preliminary_race_sex_noatt/5_1_Preliminary_NoAtt_race_binary_analysis_all_importance_weighted.parquet')
imp_sex  = load_parquet_data('outputs/5_1_preliminary_race_sex_noatt/5_1_Preliminary_NoAtt_sex_analysis_all_importance_weighted.parquet')

top_race = {'M1': get_top_features_method1(imp_race, TOP_N_LIST),
            'M3': get_top_features_method3(imp_race, TOP_N_LIST),
            'M4': get_top_features_method4(imp_race, TOP_N_LIST)}
top_sex  = {'M1': get_top_features_method1(imp_sex, TOP_N_LIST),
            'M3': get_top_features_method3(imp_sex, TOP_N_LIST),
            'M4': get_top_features_method4(imp_sex, TOP_N_LIST)}

## Save the top-N variable lists (long format)

In [ ]:
# === top_race / top_sex (M1/M3/M4 x top-N variable lists) -> long format ===
# {method: {top_n: {(ballot, model): [features]}}} transposed to one row per feature
import pandas as pd

def tops_to_long(tops_dict, target_name):
    rows = []
    for method, by_n in tops_dict.items():            # 'M1' / 'M3' / 'M4'
        for n, by_group in by_n.items():              # 10, 20, ..., 90
            for (ballot, model), feats in by_group.items():
                for rank, feat in enumerate(feats, 1):
                    rows.append({
                        'target': target_name, 'method': method, 'top_n': n,
                        'ballot': str(ballot), 'model': model, 'rank': rank, 'feature': feat,
                    })
    return pd.DataFrame(rows)

long_race = tops_to_long(top_race, 'race_binary')
long_sex  = tops_to_long(top_sex,  'sex')
long_all  = pd.concat([long_race, long_sex], ignore_index=True)

out_xlsx = "outputs/6_1_feature_selection_race_sex_noatt/6_1_top_features_long.xlsx"
os.makedirs(os.path.dirname(out_xlsx), exist_ok=True)
with pd.ExcelWriter(out_xlsx) as w:
    long_all.to_excel(w,  sheet_name='all',         index=False)
    long_race.to_excel(w, sheet_name='race_binary', index=False)
    long_sex.to_excel(w,  sheet_name='sex',         index=False)
# Also as CSV
long_all.to_csv("outputs/6_1_feature_selection_race_sex_noatt/6_1_top_features_long.csv", index=False, encoding='utf-8-sig')

print(f"Saved: {out_xlsx} / 6_1_top_features_long.csv")
print(f"  race {len(long_race)} rows, sex {len(long_sex)} rows, total {len(long_all)} rows")
print(long_all.head(8).to_string(index=False))

## Retrain with the top-N variables of each criterion

In [ ]:
N_ITERATIONS = 20

jobs = [
    ('race_binary', data_race, top_race),
    ('sex',         data_sex,  top_sex),
]

for target_col, data_dict, methods in jobs:
    for method_name, feats_all_n in methods.items():
        for n in TOP_N_LIST:
            selective_dict = build_selective_data_dict(data_dict, feats_all_n[n], target_col)
            run_analysis_for_target_weighted_selective(
                df_dict=selective_dict,
                df_name=f'6_1_FS_{method_name}_Top{n}',
                target_col=target_col,
                n_runs=N_ITERATIONS,
            )

print('All feature selection runs finished.')